# Transcriptome Layer Analysis Pipeline
This notebook contains all the scripts from the folder, organized in order.

In [ ]:
%load_ext rpy2.ipython


### `step1_download_expression.py`

In [ ]:
import GEOparse
import pandas as pd
import os

registry = pd.read_csv("manifests/expression_studies.tsv", sep="\t")

for _, row in registry.iterrows():
    
    study = row["study_id"]
    accession = row["accession"]

    print("Downloading", accession)

    gse = GEOparse.get_GEO(accession, destdir="data")

    os.makedirs(f"results/transcriptome/{study}", exist_ok=True)

    for gsm in gse.gsms:
        data = gse.gsms[gsm].table
        data.to_csv(f"results/transcriptome/{study}/expression_raw.tsv", sep="\t")
        break

### `step2_normalize_expression.py`

In [ ]:
import pandas as pd
import numpy as np

# load expression matrix
df = pd.read_csv(
    "results/transcriptome/S1/expression_raw.tsv",
    sep="\t"
)

# first column should be gene names
df = df.set_index(df.columns[0])

# convert everything to numeric
df = df.apply(pd.to_numeric, errors="coerce")

# log transform
df = np.log2(df + 1)

# z-score normalization per gene
df = df.sub(df.mean(axis=1), axis=0)
df = df.div(df.std(axis=1), axis=0)

# remove genes with NaN
df = df.dropna()

# save normalized matrix
df.to_csv(
    "results/transcriptome/S1/expression_normalized.tsv",
    sep="\t"
)

print("Normalization complete")
print("Shape:", df.shape)

### `step2_normalize_s2.py`

In [ ]:
import pandas as pd
import numpy as np
import os

# create output folder if needed
os.makedirs("results/transcriptome/S2", exist_ok=True)

# load raw expression matrix
expr = pd.read_csv(
    "data/S2/expression_raw.csv",
    low_memory=False
)

# rename first column to gene
expr = expr.rename(columns={expr.columns[0]: "gene"})

# set gene column as index
expr = expr.set_index("gene")

# log transform (log2(x+1))
expr = np.log2(expr.clip(lower=0) + 1)

# z-score normalize each gene across samples
expr_norm = expr.sub(expr.mean(axis=1), axis=0)
expr_norm = expr_norm.div(expr.std(axis=1), axis=0)

# reset index
expr_norm = expr_norm.copy()
expr_norm = expr_norm.reset_index()
expr_norm = expr_norm.reset_index()

# save normalized matrix
expr_norm.to_csv(
    "results/transcriptome/S2/expression_normalized.tsv",
    sep="\t",
    index=False
)

print("S2 normalization complete.")

### `step3_differential_expression.py`

In [ ]:
import pandas as pd
from scipy.stats import ttest_ind

expr = pd.read_csv(
    "results/transcriptome/S1/expression_normalized.tsv",
    sep="\t",
    index_col=0
)

meta = pd.read_csv(
    "results/transcriptome/S1/metadata.tsv",
    sep="\t"
)

cases = meta[meta.status=="case"]["sample"]
controls = meta[meta.status=="control"]["sample"]

results = []

for gene in expr.index:

    case_vals = expr.loc[gene, cases]
    ctrl_vals = expr.loc[gene, controls]

    stat,p = ttest_ind(case_vals, ctrl_vals)

    logfc = case_vals.mean() - ctrl_vals.mean()

    results.append([gene,logfc,p])

df = pd.DataFrame(results, columns=["gene","logFC","pvalue"])

df.to_csv(
    "results/transcriptome/S1/de.tsv",
    sep="\t",
    index=False
)

### `step3_differential_expression_R.R`

In [ ]:
%%R
library(limma)

# ----------------------------
# LOAD DATA
# ----------------------------
setwd("C:/analysis/Transcriptome layer")

expr <- read.table(
  "results/transcriptome/S2/expression_normalized.tsv",
  header = TRUE,
  sep = "\t",
  row.names = 1
)

meta <- read.table(
  "results/transcriptome/S2/metadata.tsv",
  header = TRUE,
  sep = "\t"
)

# ----------------------------
# DESIGN MATRIX
# ----------------------------
group <- factor(meta$status)
design <- model.matrix(~ group)

# ----------------------------
# FIT MODEL
# ----------------------------
fit <- lmFit(expr, design)
fit <- eBayes(fit)

# ----------------------------
# RESULTS
# ----------------------------
res <- topTable(fit, coef=2, number=Inf)

res$gene <- rownames(res)

res <- res[, c("gene","logFC","P.Value","adj.P.Val")]

write.table(
  res,
  "results/transcriptome/S2/de_limma.tsv",
  sep = "\t",
  row.names = FALSE,
  quote = FALSE
)

### `step3_differential_expression_S2.py`

In [ ]:
import pandas as pd
import scipy.stats as stats
import os

os.makedirs("results/transcriptome/S2", exist_ok=True)

# load normalized expression
expr = pd.read_csv(
    "results/transcriptome/S2/expression_normalized.tsv",
    sep="\t"
)

# load metadata
meta = pd.read_csv(
    "results/transcriptome/S2/metadata.tsv",
    sep="\t"
)

# identify case/control samples
case_samples = meta[meta["status"] == "case"]["sample"].tolist()
control_samples = meta[meta["status"] == "control"]["sample"].tolist()

results = []

for _, row in expr.iterrows():

    gene = row["gene"]

    case_vals = pd.to_numeric(row[case_samples], errors="coerce").values
    control_vals = pd.to_numeric(row[control_samples], errors="coerce").values

    if len(case_vals) == 0 or len(control_vals) == 0:
        continue

    logfc = case_vals.mean() - control_vals.mean()

    tstat, pval = stats.ttest_ind(case_vals, control_vals)

    # log fold change
    logfc = case_vals.mean() - control_vals.mean()

    # t-test
    tstat, pval = stats.ttest_ind(case_vals, control_vals)

    results.append({
        "gene": gene,
        "logFC": logfc,
        "pvalue": pval
    })

de_df = pd.DataFrame(results)

de_df.to_csv(
    "results/transcriptome/S2/de.tsv",
    sep="\t",
    index=False
)

print("S2 differential expression completed.")

### `step4_meta_analysis.py`

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import combine_pvalues

os.makedirs("results/transcriptome", exist_ok=True)

# Load DE results
s1 = pd.read_csv("results/transcriptome/S1/de.tsv", sep="\t")
s2 = pd.read_csv("results/transcriptome/S2/de.tsv", sep="\t")

# Rename columns
s1 = s1.rename(columns={
    "logFC": "logFC_S1",
    "pvalue": "pvalue_S1"
})

s2 = s2.rename(columns={
    "logFC": "logFC_S2",
    "pvalue": "pvalue_S2"
})

# Merge by gene
merged = pd.merge(s1, s2, on="gene")

results = []

for _, row in merged.iterrows():

    logfc_values = [row["logFC_S1"], row["logFC_S2"]]
    pvals = [row["pvalue_S1"], row["pvalue_S2"]]

    # meta logFC = mean
    meta_logfc = np.mean(logfc_values)

    # combine p-values (Fisher method)
    meta_pval = combine_pvalues(pvals)[1]

    direction = "up" if meta_logfc > 0 else "down"

    results.append({
        "gene": row["gene"],
        "meta_logFC": meta_logfc,
        "meta_pvalue": meta_pval,
        "direction": direction
    })

meta_df = pd.DataFrame(results)

meta_df.to_csv(
    "results/transcriptome/meta_DEG_BC.tsv",
    sep="\t",
    index=False
)

print("Meta-analysis completed.")
print("Output: results/transcriptome/meta_DEG_BC.tsv")

### `step5_coexpression_S1.py`

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs("results/networks", exist_ok=True)

# load normalized expression
expr = pd.read_csv(
    "results/transcriptome/S1/expression_normalized.tsv",
    sep="\t"
)

# gene names
genes = expr["gene"]

# expression values
expr_values = expr.drop(columns=["gene"])

# compute correlation matrix
corr = expr_values.T.corr(method="pearson")

edges = []

threshold = 0.7

for i in range(len(genes)):
    for j in range(i+1, len(genes)):

        weight = corr.iloc[i, j]

        if abs(weight) >= threshold:

            edges.append({
                "nodeA": genes.iloc[i],
                "nodeB": genes.iloc[j],
                "weight": abs(weight),
                "source": "WGCNA"
            })

edges_df = pd.DataFrame(edges)

edges_df.to_csv(
    "results/networks/coexp_S1.tsv",
    sep="\t",
    index=False
)

print("Coexpression network created.")
print("Edges:", len(edges_df))

### `step6_coexpression_network.py`

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs("results/networks", exist_ok=True)

# Load normalized expression
expr = pd.read_csv(
    "results/transcriptome/S1/expression_normalized.tsv",
    sep="\t"
)

# Load meta-DEGs
meta = pd.read_csv(
    "results/transcriptome/meta_DEG_BC.tsv",
    sep="\t"
)

deg_genes = set(meta["gene"])

# Filter expression matrix
expr = expr[expr["gene"].isin(deg_genes)]

genes = expr["gene"]
expr_values = expr.drop(columns=["gene"])

# Compute correlation
corr = expr_values.T.corr(method="pearson")

edges = []
threshold = 0.7

for i in range(len(genes)):
    for j in range(i+1, len(genes)):

        weight = corr.iloc[i, j]

        if abs(weight) >= threshold:

            edges.append({
                "nodeA": genes.iloc[i],
                "nodeB": genes.iloc[j],
                "weight": abs(weight),
                "source": "WGCNA"
            })

edges_df = pd.DataFrame(edges)

edges_df.to_csv(
    "results/networks/coexp_S1.tsv",
    sep="\t",
    index=False
)

print("Coexpression network created")
print("Edges:", len(edges_df))

### `step8_disease_enrichment.py`

In [ ]:
import pandas as pd
import gseapy as gp
import os

os.makedirs("results/enrichment", exist_ok=True)

# Load modules
modules = pd.read_csv("results/modules/final_tx_ppi_modules.tsv", sep="\t")

# Load disease PPI → background genes
ppi = pd.read_csv("results/networks/disease_ppi.tsv", sep="\t")

background_genes = set(ppi['gene1']).union(set(ppi['gene2']))
background_genes = list(background_genes)

print("Background genes:", len(background_genes))

all_results = []

for m in modules['module'].unique():
    genes = modules[modules['module'] == m]['gene'].tolist()

    # Skip tiny modules
    if len(genes) < 5:
        continue

    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=['GO_Biological_Process_2021'],
            organism='human',
            background=background_genes   # 🔥 KEY LINE
        )

        res = enr.results

        res['module_id'] = f"TX{m}"
        res = res[['module_id', 'Term', 'P-value']]

        all_results.append(res)

    except Exception as e:
        print(f"Error in module {m}: {e}")

# Combine all modules
if all_results:
    final = pd.concat(all_results, ignore_index=True)
    final.columns = ['module_id', 'term', 'pvalue']

    final.to_csv("results/enrichment/enrich_txppi_AD.tsv", sep="\t", index=False)
    print("Saved enrich_txppi_AD.tsv ✅")
else:
    print("No enrichment results generated ❌")

### `step8_louvain.py`

In [ ]:
import os
import pandas as pd
import networkx as nx
import community as community_louvain

# ✅ create folder if missing
os.makedirs("results/modules", exist_ok=True)

df = pd.read_csv("results/networks/consensus_tx.tsv", sep="\t")

G = nx.Graph()

for _, row in df.iterrows():
    G.add_edge(row['gene1'], row['gene2'], weight=row['weight'])

partition = community_louvain.best_partition(G)

modules = pd.DataFrame(list(partition.items()), columns=["gene", "module"])

modules.to_csv("results/modules/tx_modules.tsv", sep="\t", index=False)

### `step9_ppi_overlap.py`

In [ ]:
import pandas as pd

# Load
tx = pd.read_csv("results/networks/coexp_wgcna_AD.tsv", sep="\t")
ppi = pd.read_csv("results/networks/string_ppi.tsv", sep="\t")

# Normalize gene names
for df in [tx, ppi]:
    df['gene1'] = df['gene1'].str.upper().str.strip()
    df['gene2'] = df['gene2'].str.upper().str.strip()

#  KEY STEP — make edges undirected
tx['edge'] = tx.apply(lambda x: tuple(sorted([x['gene1'], x['gene2']])), axis=1)
ppi['edge'] = ppi.apply(lambda x: tuple(sorted([x['gene1'], x['gene2']])), axis=1)

# Keep only edge + weight
tx = tx[['edge', 'weight']]
ppi = ppi[['edge']]

# Merge (intersection)
merged = pd.merge(tx, ppi, on='edge')

# Recover gene1, gene2
merged[['gene1', 'gene2']] = pd.DataFrame(merged['edge'].tolist(), index=merged.index)

# Save
merged[['gene1', 'gene2', 'weight']].to_csv(
    "results/networks/intersection_tx_ppi_AD.tsv",
    sep="\t",
    index=False
)

print("Intersection edges:", len(merged))

### `step10_combine.py`

In [ ]:
import pandas as pd
tx = pd.read_csv("results/networks/consensus_tx.tsv", sep="\t")
ppi = pd.read_csv("results/networks/ppi_tx_overlap.tsv", sep="\t")

combined = pd.concat([tx, ppi], ignore_index=True)

combined.to_csv("results/networks/tx_ppi_combined.tsv", sep="\t", index=False)

### `step11_recluster.py`

In [ ]:
import os
import pandas as pd
import igraph as ig
import leidenalg

os.makedirs("results/modules", exist_ok=True)

df = pd.read_csv("results/networks/intersection_tx_ppi_AD.tsv", sep="\t")

edges = list(zip(df['gene1'], df['gene2']))
g = ig.Graph.TupleList(edges, directed=False)

partition = leidenalg.find_partition(g, leidenalg.ModularityVertexPartition)

modules = []
for i, cluster in enumerate(partition):
    for node in cluster:
        modules.append([g.vs[node]['name'], i])

modules = pd.DataFrame(modules, columns=["gene", "module"])

modules.to_csv("results/modules/modules_txppi_AD.tsv", sep="\t", index=False)

print("Leiden clustering done ✅")

### `step12_enrichment.py`

In [ ]:
import pandas as pd
import gseapy as gp
import os

os.makedirs("results/enrichment", exist_ok=True)

modules = pd.read_csv("results/modules/modules_txppi_AD.tsv", sep="\t")

for m in modules['module'].unique():
    genes = modules[modules['module'] == m]['gene'].tolist()
    
    if len(genes) < 5:
        continue  # skip tiny modules (important)
    
    enr = gp.enrichr(
        gene_list=genes,
        gene_sets=['KEGG_2021_Human', 'GO_Biological_Process_2021'],
        organism='human'
    )
    
    enr.results.to_csv(f"results/enrichment/module_{m}.tsv", sep="\t", index=False)

### `build_disease_ppi.py`

In [ ]:
import pandas as pd
import requests
import os

os.makedirs("results/networks", exist_ok=True)

# Load DEG genes
deg = pd.read_csv("results/transcriptome/meta_DEG_BC.tsv", sep="\t")
genes = deg['gene'].dropna().unique().tolist()

print("Total genes:", len(genes))

# STRING API
string_api_url = "https://string-db.org/api"
output_format = "tsv"
method = "network"

params = {
    "identifiers": "%0d".join(genes[:2000]),  # limit (STRING constraint)
    "species": 9606,
    "required_score": 700,
    "caller_identity": "chatgpt_pipeline"
}

response = requests.post(
    f"{string_api_url}/{output_format}/{method}",
    data=params
)

# Save raw
with open("results/networks/disease_ppi_raw.tsv", "w") as f:
    f.write(response.text)

# Load + clean
ppi = pd.read_csv("results/networks/disease_ppi_raw.tsv", sep="\t")

ppi = ppi[['preferredName_A', 'preferredName_B', 'score']]
ppi.columns = ['gene1', 'gene2', 'weight']

ppi.to_csv("results/networks/disease_ppi.tsv", sep="\t", index=False)

print("Saved disease_ppi.tsv ✅")

### `check_s2_expression.py`

In [ ]:
import pandas as pd

expr = pd.read_csv(
    "results/transcriptome/S2/expression_raw.tsv",
    sep="\t"
)

print(expr.shape)
print(expr.columns[:20])

### `clean_expression.py`

In [ ]:
import pandas as pd

df = pd.read_csv("results/transcriptome/S2/expression_normalized.tsv", sep="\t")

df = df.drop(columns=["index"])
df = df.set_index("gene")

df.to_csv("results/transcriptome/S2/expression_for_wgcna.tsv", sep="\t")

print("File cleaned ✅")

### `convert_s2_expression.py`

In [ ]:
import pandas as pd

expr = pd.read_csv(
    "data/S2/expression_raw.csv"
)

expr.rename(columns={expr.columns[0]: "gene"}, inplace=True)

expr.to_csv(
    "results/transcriptome/S2/expression_raw.tsv",
    sep="\t",
    index=False
)

print("S2 expression file fixed.")
print("Shape:", expr.shape)

### `extract_edges.R`

In [ ]:
%%R
# Load adjacency matrix
load("results/networks/adjacency.RData")

print("Adjacency loaded")

genes <- colnames(adj)

edges <- data.frame()
count <- 0

print("Extracting edges...")

for (i in 1:(ncol(adj)-1)) {
  for (j in (i+1):ncol(adj)) {
    
    val <- adj[i, j]
    
    # ✅ NA-safe + LOWER threshold (important fix)
    if (!is.na(val) && val > 0.2) {
      edges <- rbind(edges,
                     data.frame(
                       gene1 = genes[i],
                       gene2 = genes[j],
                       weight = val
                     ))
      count <- count + 1
    }
  }
}

print(paste("Total edges:", count))

# Save edge list
write.table(edges,
            file = "results/networks/coexp_wgcna_AD.tsv",
            sep = "\t",
            row.names = FALSE,
            quote = FALSE)

print("Network saved as coexp_wgcna_AD.tsv ✅")

### `filter_genes.py`

In [ ]:
import pandas as pd

df = pd.read_csv("results/transcriptome/S2/de.tsv", sep="\t")

# Keep only valid gene-like names (uppercase letters/numbers)
df = df[df['gene'].str.match(r'^[A-Z0-9]+$')]

# Remove antisense / weird ones again just in case
df = df[~df['gene'].str.contains(r'-AS|^MT-|^5S_|^7SK|^7SL')]

# Drop NA
df = df[df['gene'].notna()]

df.to_csv("results/transcriptome/S2/de_filtered.tsv", sep="\t", index=False)

print("Strict filtering done ✅")

### `geneoverlap.py`

In [ ]:
import pandas as pd

tx = pd.read_csv("results/networks/coexp_wgcna_AD.tsv", sep="\t")
ppi = pd.read_csv("results/networks/disease_ppi.tsv", sep="\t")

tx_genes = set(tx['gene1']).union(set(tx['gene2']))
ppi_genes = set(ppi['gene1']).union(set(ppi['gene2']))

overlap = tx_genes.intersection(ppi_genes)

print("TX genes:", len(tx_genes))
print("PPI genes:", len(ppi_genes))
print("Overlap:", len(overlap))

### `inspect_s2_expression.py`

In [ ]:
import pandas as pd

file_path = "data/S2/expression_raw.csv"

# read only header
df = pd.read_csv(file_path, nrows=0)

print(df.columns.tolist())

### `prepare_expression_matrix.py`

In [ ]:
import pandas as pd
import os

os.makedirs("results/transcriptome/S1", exist_ok=True)

# load GEO expression matrix
df = pd.read_csv(
    "data/GSE81538_gene_expression_405_transformed.csv"
)

# first column is gene symbol
df = df.rename(columns={df.columns[0]: "gene"})

# save as pipeline format
df.to_csv(
    "results/transcriptome/S1/expression_raw.tsv",
    sep="\t",
    index=False
)

print("Expression matrix prepared.")

### `prepare_string_ppi.py`

In [ ]:
import pandas as pd

ppi = pd.read_csv("data/ppi/string_ppi.tsv", sep="\t")

# Keep strong interactions only
ppi = ppi[ppi['combined_score'] > 700]

# Rename columns
ppi = ppi.rename(columns={
    'protein1': 'gene1',
    'protein2': 'gene2'
})

# Keep only required columns
ppi = ppi[['gene1', 'gene2']]

# Normalize gene names
ppi['gene1'] = ppi['gene1'].str.upper().str.strip()
ppi['gene2'] = ppi['gene2'].str.upper().str.strip()

# Save
ppi.to_csv("results/networks/string_ppi.tsv", sep="\t", index=False)

print("STRING PPI ready ✅")

### `wgcna_network.R`

In [ ]:
%%R
# ==============================
# WGCNA FULL PIPELINE (ALL GENES)
# ==============================

library(WGCNA)
options(stringsAsFactors = FALSE)
allowWGCNAThreads()

# ------------------------------
# STEP 1 — Load data
# ------------------------------
expr <- read.table(
  "results/transcriptome/S2/expression_for_wgcna.tsv",
  header = TRUE,
  sep = "\t",
  row.names = 1
)

# transpose → WGCNA needs samples as rows
datExpr <- t(expr)

# ------------------------------
# STEP 2 — Check good genes/samples
# ------------------------------
gsg <- goodSamplesGenes(datExpr, verbose = 3)

if (!gsg$allOK) {
  datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes]
}

# ------------------------------
# STEP 3 — Sample clustering (QC)
# ------------------------------
sampleTree <- hclust(dist(datExpr), method = "average")

pdf("results/networks/sample_clustering.pdf")
plot(sampleTree, main = "Sample clustering")
dev.off()

# ------------------------------
# STEP 4 — Soft-threshold selection
# ------------------------------
powers <- c(1:20)

sft <- pickSoftThreshold(datExpr,
                         powerVector = powers,
                         verbose = 5)

pdf("results/networks/soft_threshold.pdf")

par(mfrow = c(1,2))

# scale-free topology
plot(sft$fitIndices[,1],
     -sign(sft$fitIndices[,3]) * sft$fitIndices[,2],
     xlab = "Soft Threshold (power)",
     ylab = "Scale Free Topology Model Fit",
     type = "n")

text(sft$fitIndices[,1],
     -sign(sft$fitIndices[,3]) * sft$fitIndices[,2],
     labels = powers,
     col = "red")

abline(h = 0.9, col = "blue")

# mean connectivity
plot(sft$fitIndices[,1],
     sft$fitIndices[,5],
     xlab = "Soft Threshold (power)",
     ylab = "Mean Connectivity",
     type = "n")

text(sft$fitIndices[,1],
     sft$fitIndices[,5],
     labels = powers,
     col = "red")

dev.off()

# ------------------------------
# STEP 5 — Choose power
# ------------------------------
softPower <- 6   # adjust based on plot (usually 5–8)

# ------------------------------
# STEP 6 — Adjacency matrix
# ------------------------------
adjacency <- adjacency(datExpr,
                       power = softPower,
                       type = "signed")

save(adjacency,
     file = "results/networks/adjacency_full.RData")

# ------------------------------
# STEP 7 — TOM matrix
# ------------------------------
TOM <- TOMsimilarity(adjacency)
dissTOM <- 1 - TOM

# ------------------------------
# STEP 8 — Gene clustering
# ------------------------------
geneTree <- hclust(as.dist(dissTOM), method = "average")

pdf("results/networks/gene_clustering.pdf")
plot(geneTree, main = "Gene clustering (TOM)")
dev.off()

# ------------------------------
# STEP 9 — Module detection
# ------------------------------
dynamicMods <- cutreeDynamic(
  dendro = geneTree,
  distM = dissTOM,
  deepSplit = 2,
  pamRespectsDendro = FALSE,
  minClusterSize = 30
)

moduleColors <- labels2colors(dynamicMods)

pdf("results/networks/modules_colors.pdf")
plotDendroAndColors(geneTree,
                    moduleColors,
                    "Modules",
                    dendroLabels = FALSE)
dev.off()

# ------------------------------
# STEP 10 — Module eigengenes
# ------------------------------
MEs <- moduleEigengenes(datExpr,
                        colors = moduleColors)$eigengenes

# ------------------------------
# STEP 11 — Save module assignments
# ------------------------------
modules <- data.frame(
  Gene = colnames(datExpr),
  Module = moduleColors
)

write.table(modules,
            "results/modules/modules_wgcna_full.tsv",
            sep = "\t",
            row.names = FALSE,
            quote = FALSE)

# ------------------------------
# STEP 12 — Export coexpression edges
# ------------------------------
# threshold for edge export
threshold <- 0.2

edges <- which(adjacency > threshold, arr.ind = TRUE)

edge_list <- data.frame(
  Gene1 = colnames(datExpr)[edges[,1]],
  Gene2 = colnames(datExpr)[edges[,2]],
  Weight = adjacency[edges]
)

# remove self loops
edge_list <- edge_list[edge_list$Gene1 != edge_list$Gene2, ]

write.table(edge_list,
            "results/networks/coexp_wgcna_full.tsv",
            sep = "\t",
            row.names = FALSE,
            quote = FALSE)